In [1]:
import pandas as pd, numpy as np, dendropy

df = pd.read_csv("../data/guiana_teleo_clean.csv")
df = df[df["in_tree"]]
species = sorted(df["species_name"].unique())          # 271 species
print("species:", len(species))

species: 271


In [2]:
# ---------- 1- TREE distance matrix ----------
tree = dendropy.Tree.get(path="../data/fish_tree_full.tre", schema="newick",
                         preserve_underscores=True)
labels = [s.replace(" ", "_") for s in species]
sub = tree.extract_tree_with_taxa_labels(labels=labels)   # keep only our 271 species
pdm = sub.phylogenetic_distance_matrix()
taxa = {t.label.strip("'").replace("_", " "): t for t in sub.taxon_namespace}

n = len(species)
D_tree = np.zeros((n, n))
for i, a in enumerate(species):
    for j in range(i + 1, n):
        d = pdm.patristic_distance(taxa[a], taxa[species[j]])
        D_tree[i, j] = D_tree[j, i] = d
D_tree /= D_tree.max()                                     # scale to 0–1
print("tree matrix:", D_tree.shape, "| min off-diag:", D_tree[D_tree > 0].min().round(4))

tree matrix: (271, 271) | min off-diag: 0.0007


In [3]:
# ---------- 2- TAXONOMY distance matrix ----------
info = df.drop_duplicates("species_name").set_index("species_name")[["Genus", "Family", "Order"]]
def tax_dist(a, b):
    if a == b: return 0
    if info.at[a, "Genus"]  == info.at[b, "Genus"]:  return 1
    if info.at[a, "Family"] == info.at[b, "Family"]: return 2
    if info.at[a, "Order"]  == info.at[b, "Order"]:  return 3
    return 4
D_tax = np.array([[tax_dist(a, b) for b in species] for a in species]) / 4.0

In [4]:
# ---------- save ----------
np.savez("../data/distance_matrices.npz", species=np.array(species),
         D_tree=D_tree, D_tax=D_tax)
print("saved ../data/distance_matrices.npz")

saved ../data/distance_matrices.npz


In [6]:
import numpy as np
z = np.load("../data/distance_matrices.npz")
print(z.files)            # ['species', 'D_tree', 'D_tax']
print(z["D_tree"][:5, :5])

['species', 'D_tree', 'D_tax']
[[0.         0.60202492 0.60202493 0.92060729 0.26597377]
 [0.60202492 0.         0.44347226 0.92060728 0.60202492]
 [0.60202493 0.44347226 0.         0.92060729 0.60202492]
 [0.92060729 0.92060728 0.92060729 0.         0.92060729]
 [0.26597377 0.60202492 0.60202492 0.92060729 0.        ]]
